# Quantum Bayes — Detailed Notes (Session 18)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller

> **Purpose.** These notes turn the slide bullets into a stand-alone reference on *quantum* Bayesian modeling: how to encode priors/likelihoods into circuits, estimate probabilities with **quantum amplitude estimation (QAE)**, sketch “quantum Bayesian networks,” and build small uncertainty-aware pipelines in Qiskit. We focus on NISQ-practical patterns and avoid hype.

---

## Session road-map
1. Recap: QGNNs → why probabilistic reasoning  
2. Two tracks for “Quantum Bayes”  
   - (A) **Quantum-accelerated Bayes**: classical models + quantum speedups for *probability estimation*  
   - (B) **Quantum probabilistic models**: “Bayesian networks” with quantum states/channels  
3. Encoding priors & conditionals as **state preparation** (q-sampling)  
4. Inference via **QAE** (phase-estimation-based) vs. shot sampling  
5. Tiny quantum Bayes nets (2–3 nodes) in Qiskit  
6. Uncertainty quantification (UQ) use-cases & hybrid loops  
7. Comparisons, caveats, and what’s realistic on NISQ  
8. Mini-exercises & lab outline

---

## 0) Recap → why Bayes, why quantum?
- QGNNs handled **structured** signals. Many AI tasks still need **uncertainty**: decisions under sparse data, safety, finance, molecules.  
- Classical Bayes recap: $P(A\mid B)=\dfrac{P(B\mid A)P(A)}{P(B)}$. In practice, we estimate normalisers and conditionals via **sampling** or **variational** methods.  
- **Quantum angle:** circuits can *prepare* distributions and **estimate probabilities** with **quadratic** sample complexity improvement via **QAE** (ideal setting).

---

## 1) Two meanings of “Quantum Bayes”

### (A) Quantum-accelerated Bayesian inference (most NISQ-practical)
- Keep the **classical** Bayesian model (BN, HMM, Bayes net).  
- Use a circuit to **prepare** success/failure amplitudes encoding a Bernoulli event of interest (e.g., $B$ occurs), then run **QAE** to estimate $p=\Pr(B)$ with $O(1/\epsilon)$ oracle calls vs $O(1/\epsilon^2)$ classical Monte Carlo.  
- Apply Bayes’ rule using quantum-estimated pieces.

### (B) Quantum Bayesian networks (qBNs) / quantum generative models (researchy)
- Nodes carry **quantum states** (pure or mixed); edges are **quantum channels/unitaries** encoding “conditional” structure.  
- The joint “distribution” is a **density operator**; inference becomes measurement + channel composition.  
- Related lines: **QCBMs** (Born machines), **quantum Boltzmann machines**, tensor-network generative models.

> **Takeaway:** (A) is the robust way to get a speedup (under oracle access). (B) is conceptually rich but early-stage and small-scale on NISQ.

---

## 2) From probabilities to circuits (q-sampling prep)

### Single Bernoulli
Given $p\in[0,1]$, set $\theta=2\arcsin(\sqrt{p})$.  
$$
R_y(\theta)\,|0\rangle=\sqrt{1-p}\,|0\rangle+\sqrt{p}\,|1\rangle.
$$
Measuring in Z yields $\Pr(1)=p$.

### Conditional Bernoulli
Model $X\to Y$ with conditionals $p_1=\Pr(Y{=}1\!\mid X{=}1)$, $p_0=\Pr(Y{=}1\!\mid X{=}0)$.  
Prepare $X$ with prior $p_X$, then apply **controlled-$R_y$** on $Y$ with angles set by $p_0,p_1$.

### 3-node toy (Cloudy→Rain→WetGrass)
Encode priors and $P(\text{Rain}\mid\text{Cloudy})$, $P(\text{Wet}\mid\text{Rain})$ as cascaded controlled rotations. Sampling the 3 qubits gives joint counts; posteriors follow by classical conditioning.

> **Note on scale.** State prep for arbitrary discrete distributions may be costly; for tiny nets (2–4 qubits) it’s straightforward and ideal for labs.

---

## 3) Inference with **Quantum Amplitude Estimation (QAE)**

**Goal.** Estimate $a=\Pr(\text{“good”})$ encoded as amplitude of $|1\rangle$ on a flag qubit after a unitary $A$:  
$$
A|0\dots 0\rangle=\sqrt{1-a}\,| \text{bad}\rangle|0\rangle+\sqrt{a}\,|\text{good}\rangle|1\rangle.
$$

**Why QAE.** Quadratic improvement in sample complexity under ideal oracles: $\tilde{O}(1/\epsilon)$ calls (vs. $O(1/\epsilon^2)$ shots).

**How.** QAE is built on **phase estimation** of the Grover iterate; NISQ-friendly variants (e.g., iterative / maximum-likelihood AE) avoid deep QFT.

**Use in Bayes.** Encode pieces like $P(B)$, $P(B\mid A)$, priors. Estimate each with QAE, then compute the posterior $P(A\mid B)$ classically.

> **Reality check.** On noisy hardware, the asymptotic advantage can be eroded; **iterative AE** with error-mitigation is the pragmatic choice.

---

## 4) Qiskit patterns (primitives-based)

> **APIs evolve.** Prefer **primitives**: `Sampler` for bitstring probabilities, `Estimator` for expectations. For amplitude estimation, use the algorithms in `qiskit.algorithms` (e.g., `IterativeAmplitudeEstimation`, `EstimationProblem`) where available; otherwise roll a small iterative AE.

### A) Tiny 2-node Bayes net (prior + conditional) — sampling baseline
```python
# pip install qiskit qiskit-aer
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler
import numpy as np

# Encode prior P(X=1)=px and conditional P(Y=1|X)
px = 0.3
p1, p0 = 0.8, 0.2  # P(Y=1|X=1), P(Y=1|X=0)

theta_x = 2*np.arcsin(np.sqrt(px))
theta_y1 = 2*np.arcsin(np.sqrt(p1))
theta_y0 = 2*np.arcsin(np.sqrt(p0))

qc = QuantumCircuit(2)  # q0=X (parent), q1=Y (child)
qc.ry(theta_x, 0)
qc.cry(theta_y1, 0, 1)      # when X=1
qc.x(0); qc.cry(theta_y0, 0, 1); qc.x(0)  # when X=0 (decompose controlled-on-0)

# Query posterior P(X=1|Y=1) from samples
sampler = Sampler()
shots = 20_000
res = sampler.run([qc], shots=shots).result()
counts = res.quasi_dists[0].nearest_probability_distribution().binary_probabilities()
# accumulate counts:
import collections
c = collections.Counter()
for bitstring, p in counts.items():
    # bitstring like 'yx'
    y = int(bitstring[0]); x = int(bitstring[1])
    c[(x,y)] += p*shots

p_y1 = (c[(0,1)] + c[(1,1)]) / shots
p_x1y1 = c[(1,1)] / shots / p_y1
print("Estimated P(Y=1)≈", round(p_y1,3), "  Posterior P(X=1|Y=1)≈", round(p_x1y1,3))
```

### B) Iterative Amplitude Estimation (conceptual sketch)
```python
# Conceptual: wrap the above qc with an ancilla marking event Y=1 as "good"
# and build an EstimationProblem for IterativeAmplitudeEstimation (IAE).
# In newer Qiskit, you can do something like:

from qiskit.algorithms import EstimationProblem, IterativeAmplitudeEstimation
from qiskit.circuit import QuantumCircuit

# Build A that prepares the flag qubit 'good' iff Y=1
A = QuantumCircuit(2, name='A')
A.compose(qc, inplace=True)   # qc prepares joint X,Y with desired probs
# We "treat" Y as the success qubit (index 1 here)
problem = EstimationProblem(
    state_preparation=A,
    objective_qubits=[1]  # measure Y
)

iae = IterativeAmplitudeEstimation(epsilon_target=0.02, alpha=0.05)  # 95% conf.
result = iae.estimate(problem)
print("QAE estimate of P(Y=1):", result.estimation)
```
> If the exact class names differ in your Qiskit version, the **idea** remains: define an `EstimationProblem` with a flag qubit, run an iterative AE variant to estimate that probability with fewer oracle calls than plain sampling.

---

## 5) Quantum Bayesian “networks” (qBN) — what that means here

### Minimal, NISQ-friendly interpretation
- Treat each conditional $P(\text{child}\mid\text{parents})$ as a **controlled rotation** layer.  
- The whole net becomes a **state-preparation circuit** producing Born probabilities over the node bits.  
- **Learning** = optimise the rotation angles to fit observed data (maximum likelihood on counts).

### Beyond NISQ (research)
- Replace conditionals by **quantum channels** (CPTP maps); nodes carry **density matrices**.  
- Interference allows non-classical correlations but complicates semantics and training.

**Pitfalls**
- Parameter growth with many parents (multi-control gates).  
- Order and layout matter; decoherence blurs “probabilities”.

---

## 6) Uncertainty quantification (UQ) & hybrid loops

**Where it helps today**
- **Small-scale** Bayes nets / reliability models requiring many probability queries (risk, toy medical diagnosis).  
- **Posterior expectations** via QAE (e.g., $ \mathbb{E}[f(X)] $ where $f$ is encoded as a phase/flag).  
- **Bayesian neural networks**: put a quantum sampler/QAE in the inner loop for certain likelihood integrals (research).

**Hybrid loop**
1. Classical model defines the structure.  
2. For each probability query needed in inference/learning, call a **quantum estimator** (Sampler or QAE).  
3. Update beliefs **classically**; iterate.

---

## 7) Classical vs. Quantum Bayes — honest comparison

| Aspect | Classical Bayes | Quantum Bayes (NISQ) |
|---|---|---|
| Probability est. | Monte Carlo $O(1/\epsilon^2)$ | QAE $O(1/\epsilon)$ (ideal); iterative AE on NISQ |
| Model scale | Large BNs with factor graphs, VI/MCMC | Tiny nets (2–6 qubits) practical today |
| Robustness | Mature tooling (PyMC/Stan) | Noise & calibration sensitive |
| Expressivity | Classical probs | Can model **non-factorizable** Born distributions |
| Best use | Big real systems | Small nets; inner loops where AE helps |

> **Bottom line:** promise exists when **probability queries dominate** cost and oracles can be built efficiently. Otherwise, classical methods win on scale and maturity.

---

## 8) Practical guidance & pitfalls

- **Angles from probabilities:** clamp $p$ to $[10^{-6}, 1-10^{-6}]$ before $\arcsin$ to avoid NaNs.  
- **Layout-aware controls:** multi-controlled rotations transpile into many CNOTs; prefer factoring or ancilla-assisted decompositions.  
- **Iterative AE over textbook AE:** fewer qubits/depth (no QFT), more NISQ-friendly.  
- **Calibration:** readout error mitigation crucial for small probabilities.  
- **Composability:** reuse compiled subcircuits for repeated conditionals.  
- **Validation:** compare quantum estimates to classical Monte Carlo on the **same** toy net.

---

## 9) Worked toy example — 3-node weather net

**Structure:** Cloudy $C$ → Rain $R$ → Wet $W$. Given $p_C,\, p(R{=}1\mid C),\, p(W{=}1\mid R)$.  
1. Prepare $C$ with $R_y(2\arcsin\sqrt{p_C})$.  
2. Controlled-$R_y$ from $C$ to $R$ for both branches ($C{=}0/1$).  
3. Controlled-$R_y$ from $R$ to $W$.  
**Queries:** $P(W{=}1)$ via Sampler/QAE; posterior $P(C{=}1\mid W{=}1)$ via counts or Bayes update.

---

## 10) Mini-exercises (answers in Appendix)

1. **Angles:** Show that choosing $\theta=2\arcsin(\sqrt{p})$ yields $\Pr(1)=p$ after $R_y(\theta)$.  
2. **Controlled conditional:** Derive a controlled-on-0 rotation using X sandwiches around a standard control-on-1 rotation.  
3. **AE vs sampling:** For target precision $\epsilon=0.02$, estimate the relative #oracle calls for QAE vs classical Monte Carlo (asymptotically).  
4. **Transpilation budget:** Your 3-node net uses 6 controlled rotations. Assuming each decomposes into ≈3 CNOTs on your backend, estimate added two-qubit error if per-CNOT error is 1%. Suggest two mitigations.  
5. **Learning angles:** Given observed counts for a 2-node net, write the MLE for $\hat{p}_X$, $\hat{p}_1=\Pr(Y{=}1\mid X{=}1)$, $\hat{p}_0=\Pr(Y{=}1\mid X{=}0)$ and map them to circuit angles.

---

## 11) Summary (Session 18)
- **Quantum-accelerated Bayes** = classical models + **QAE**/**Sampler** for probability queries (most practical now).  
- **qBNs** as full quantum graphical models are promising but small-scale on NISQ.  
- Use **controlled rotations** to encode conditionals; verify against classical baselines.  
- Prefer **iterative amplitude estimation** + mitigation; be topology-aware to keep depth low.  
- Real wins come when probability estimation dominates and oracles are efficient.

---

## 12) Looking ahead
- **Next Session:** Quantum K-Means — distance estimation with swap tests & amplitude encoding; hybrid clustering pipelines.  
- **Homework 5 (Quantum Bayes):**  
  1) Build the 3-node weather net; estimate $P(W{=}1)$ via Sampler vs Iterative AE (same shots budget); report RMSE.  
  2) Learn conditional angles from synthetic counts (MLE) and validate posteriors.  
  3) (Bonus) Add readout calibration and show its effect for $p\in\{0.05, 0.95\}$.

---

## Appendix — solutions (sketch)

1. $R_y(\theta)|0\rangle=\cos(\tfrac\theta2)|0\rangle+\sin(\tfrac\theta2)|1\rangle$. Set $\sin^2(\tfrac\theta2)=p\Rightarrow \theta=2\arcsin\sqrt p$.  
2. Control-on-0 of $U$: apply $X$ on control, then controlled-$U$, then $X$ on control.  
3. Asymptotically, classical MC needs $O(1/\epsilon^2)\approx 2500$ samples; QAE $O(1/\epsilon)\approx 50$ oracle calls (constants ignored).  
4. 6 rotations × ~3 CNOTs ≈ 18 CNOTs → multiplicative shrink $(1-0.01)^{18}\approx 0.83$. Mitigate via (i) transpiler level-2/3 with layout optimisation, (ii) fewer entanglers or ancilla-assisted controlled rotations, plus readout mitigation.  
5. From counts: $\hat p_X=\frac{n_{X=1}}{n}$; $\hat p_1=\frac{n_{Y=1,X=1}}{n_{X=1}}$; $\hat p_0=\frac{n_{Y=1,X=0}}{n_{X=0}}$. Map $\theta_X=2\arcsin\sqrt{\hat p_X}$, $\theta_{Y|X=1}=2\arcsin\sqrt{\hat p_1}$, $\theta_{Y|X=0}=2\arcsin\sqrt{\hat p_0}$.
